# Creazione File

Import delle librerie  

In [2]:
import mne
import numpy as np
import pandas as pd
import os
import h5py
import gc # Garbage Collector per la pulizia forzata della memoria RAM
from scipy.signal import welch

In [3]:
DB_PATH = "../data/processed/eeg_dataset.h5"

## File intero segmentato

Nella prima parte del processo viene generato il dataset.

Si parte da 9 file .cnt, ognuno contenente registrazioni EEG della durata di circa 3000 secondi. Poiché le registrazioni originarie comprendono 66 canali, abbiamo escluso 4 elettrodi non rilevanti, mantenendone solo 62.

Ognuno dei file rappresenta i segnali EEG di un determinato utente in una specifica sessione (il setup prevede 3 utenti per 3 sessioni ciascuno).

Durante ogni sessione, agli utenti vengono mostrati 15 video scelti per suscitare specifiche emozioni. Tra un video e l'altro, ai partecipanti viene dato il tempo di assegnare uno `score` (da 1 a 5) che rappresenta l'intensità con cui hanno percepito quell'emozione.

In fase di estrazione, abbiamo selezionato esclusivamente i segmenti temporali in cui i segnali coincidevano con la visione dei video. Questi segmenti sono stati suddivisi in finestre non sovrapposte da 1.5 secondi. Dato che la frequenza di campionamento è di 1000 Hz, per ogni finestra temporale otteniamo una matrice dove 62 sensori registrano 1500 campioni ciascuno.

A ogni finestra estratta vengono associati dei metadati: l'etichetta dell'emozione target (label), lo score di autovalutazione, il session_ID e il person_ID (quest'ultimi utili per eventuali test di validazione incrociata tra pazienti).
In conclusione, ogni record del dataset finale in input alla rete è costituito da una matrice di dimensioni $62 \times 1500$.

In [4]:

# ==========================================
# 1. CONFIGURAZIONE PARAMETRI E COSTANTI
# ==========================================
SFREQ = 1000 # Frequenza di campionamento (Hz)
WINDOW_SEC = 1.5 # Lunghezza della finestra temporale in secondi
WINDOW_SIZE = int(WINDOW_SEC * SFREQ) # Risultato: 1500 campioni per ogni segmento
USELESS_CH = ['M1', 'M2', 'VEO', 'HEO'] # Canali non EEG da scartare per evitare rumore elettrooculografico

# ==========================================
# 2. DIZIONARI DEI METADATI SPERIMENTALI
# ==========================================
# Marcatori temporali (inizio e fine in secondi) per i 15 video di ogni sessione
TIMESTAMPS = {
    1: {'start': [30, 132, 287, 555, 773, 982, 1271, 1628, 1730, 2025, 2227, 2435, 2667, 2932, 3204],
        'end':   [102, 228, 524, 742, 920, 1240, 1568, 1697, 1994, 2166, 2401, 2607, 2901, 3172, 3359]},
    2: {'start': [30, 299, 548, 646, 836, 1000, 1091, 1392, 1657, 1809, 1966, 2186, 2333, 2490, 2741],
        'end':   [267, 488, 614, 773, 967, 1059, 1331, 1622, 1777, 1908, 2153, 2302, 2428, 2709, 2817]},
    3: {'start': [30, 353, 478, 674, 825, 908, 1200, 1346, 1451, 1711, 2055, 2307, 2457, 2726, 2888],
        'end':   [321, 418, 643, 764, 877, 1147, 1284, 1418, 1679, 1996, 2275, 2425, 2664, 2857, 3066]}
}

# NUOVA MAPPATURA ETICHETTE (Codifica: 0=Disgust, 1=Fear, 2=Sad, 3=Neutral, 4=Happy)
LABELS_MAP = {
    1: [4, 1, 3, 2, 0] * 3,
    2: [2, 1, 3, 0, 4] + [4, 0, 3, 2, 1] + [3, 4, 1, 2, 0],
    3: [2, 1, 3, 0, 4] + [4, 0, 3, 2, 1] + [3, 4, 1, 2, 0]
}

# TRASCRIZIONE DEGLI SCORE DI AUTOVALUTAZIONE (0-5)
# Struttura: SCORES_MAP[Paziente][Sessione] = [Lista di 15 score]
SCORES_MAP = {
    1: { 
        1: [2, 4, 5, 2, 1, 4, 5, 4, 3, 2, 4, 4, 5, 5, 2],
        2: [3, 2, 5, 3, 4, 3, 5, 4, 4, 3, 5, 2, 5, 5, 2],
        3: [3, 2, 5, 5, 2, 5, 3, 4, 2, 5, 5, 3, 5, 4, 5]
    },
    2: { 
        1: [3, 3, 4, 4, 4, 3, 5, 5, 4, 5, 4, 4, 4, 5, 5],
        2: [4, 2, 5, 4, 4, 4, 3, 5, 5, 4, 4, 5, 5, 4, 1],
        3: [4, 4, 5, 5, 3, 5, 2, 5, 4, 5, 5, 5, 5, 5, 1]
    },
    3: { 
        1: [4, 3, 4, 3, 4, 5, 5, 5, 5, 4, 5, 4, 5, 5, 4],
        2: [4, 3, 5, 3, 4, 4, 5, 4, 5, 4, 5, 5, 5, 5, 4],
        3: [5, 4, 5, 5, 4, 4, 4, 5, 5, 5, 4, 5, 5, 5, 5]
    }
}

FILE_LIST = [
    "1_1_20180804.cnt", "1_2_20180810.cnt", "1_3_20180808.cnt",
    "2_1_20180416.cnt", "2_2_20180419.cnt", "2_3_20180425.cnt",
    "3_1_20180414.cnt", "3_2_20180419.cnt", "3_3_20180424.cnt"
]

h5_filename = DB_PATH

# ==========================================
# 3. CREAZIONE ARCHIVIO HDF5 E PIPELINE DI ESTRAZIONE
# ==========================================
with h5py.File(h5_filename, 'w') as hf:
    # Inizializzazione dei tensori su disco con dimensione estendibile.
    # y_ds ora ha shape (0, 4) per ospitare lo Score.
    X_ds = hf.create_dataset('X', shape=(0, 62, 1500), maxshape=(None, 62, 1500), dtype='float32', chunks=True)
    y_ds = hf.create_dataset('y', shape=(0, 4), maxshape=(None, 4), dtype='int8')
    
    current_idx = 0 # Puntatore per tracciare la riga di scrittura sul disco
    
    for file_name in FILE_LIST:
        
        path_file = "../data/raw/"+file_name 
       
        if not os.path.exists(path_file):
            continue # Salta il ciclo se il file .cnt manca nella directory
            
        print(f"Estrazione e salvataggio in corso: {path_file}")
        
        # Parsificazione del nome file per dedurre Paziente e Sessione
        parts = file_name.split('_')
        p_id = int(parts[0])
        s_id = int(parts[1])
        
        # Apertura file in modalità "Lazy" (preload=False) per non caricare 50 minuti di EEG in RAM
        raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)
        raw.drop_channels(USELESS_CH, on_missing='ignore')
        
        # Estrazione metadati operativi per il file corrente
        starts, ends = TIMESTAMPS[s_id]['start'], TIMESTAMPS[s_id]['end']
        labels = LABELS_MAP[s_id]
        scores = SCORES_MAP[p_id][s_id] # Estrazione della lista dei 15 score specifici
        
        file_segments = []
        file_meta = []
        
        # Iterazione sui 15 stimoli video
        for i in range(15):
            v_start = starts[i] * SFREQ
            v_end = ends[i] * SFREQ
            
            # MNE estrae dal disco solo la porzione esatta di segnale compresa tra v_start e v_end
            video_data = raw.get_data(start=v_start, stop=v_end).astype(np.float32)
            
            # Calcolo del numero di segmenti interi da 1.5s ricavabili da questo video
            n_windows = video_data.shape[1] // WINDOW_SIZE
            
            for w in range(n_windows):
                w_start = w * WINDOW_SIZE
                w_end = w_start + WINDOW_SIZE
                
                # Taglio del segmento e accodamento alla lista temporanea
                file_segments.append(video_data[:, w_start:w_end]) #!da controllare
                
                # Inserimento dei metadati completi: Paziente, Sessione, Emozione Target, Autovalutazione
                file_meta.append([p_id, s_id, labels[i], scores[i]])
                
        # --- FASE DI SCRITTURA SU DISCO ---
        n_new_samples = len(file_segments)
        
        # Espansione della struttura del file H5
        X_ds.resize((current_idx + n_new_samples, 62, 1500))
        y_ds.resize((current_idx + n_new_samples, 4))
        
        # Scrittura effettiva dei blocchi NumPy sul disco rigido
        X_ds[current_idx : current_idx + n_new_samples] = np.array(file_segments, dtype=np.float32)
        y_ds[current_idx : current_idx + n_new_samples] = np.array(file_meta, dtype=np.int8)
        
        # Aggiornamento puntatore per il file successivo
        current_idx += n_new_samples
        
        # --- PULIZIA DELLA RAM ---
        # Rimozione esplicita delle variabili pesanti allocate in questo ciclo
        del raw, file_segments, file_meta, video_data
        gc.collect() # Invocazione forzata del Garbage Collector

print(f"\nGenerazione del dataset HDF5 completata: {h5_filename}")
print(f"Campioni totali estratti: {current_idx}")
print("Struttura metadati (y): [Patient_ID, Session_ID, Label, Score]")

Estrazione e salvataggio in corso: ../data/raw/1_1_20180804.cnt
Estrazione e salvataggio in corso: ../data/raw/1_2_20180810.cnt
Estrazione e salvataggio in corso: ../data/raw/1_3_20180808.cnt
Estrazione e salvataggio in corso: ../data/raw/2_1_20180416.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_27944\2272582560.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/2_2_20180419.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_27944\2272582560.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/2_3_20180425.cnt


...

Estrazione e salvataggio in corso: ../data/raw/3_2_20180419.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_27944\2272582560.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/3_3_20180424.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_27944\2272582560.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)



Generazione del dataset HDF5 completata: ../data/processed/eeg_dataset.h5
Campioni totali estratti: 14691
Struttura metadati (y): [Patient_ID, Session_ID, Label, Score]


# Pre Processing

Nella seconda fase del processo viene eseguito il preprocessing e la pulizia del dataset precedentemente generato. Poiché l'obiettivo è addestrare il modello su segnali in cui l'emozione target è stata effettivamente provata in modo netto, abbiamo introdotto un filtro di confidenza basato sull'autovalutazione degli utenti. 

Nello specifico, sono stati scartati tutti i campioni EEG associati a uno score inferiore a 3 (su una scala da 1 a 5), mantenendo esclusivamente le finestre temporali relative a risposte emotive di intensità medio-alta. 

Per ragioni di efficienza computazionale e per non saturare la memoria RAM del sistema (dato il notevole peso dei tensori), l'estrazione dei dati puliti avviene in due step. Inizialmente, il sistema legge in memoria solo la matrice dei metadati per individuare e mappare gli indici esatti dei segmenti "validi" (con score $\ge 3$). Successivamente, viene creato un nuovo file HDF5 di destinazione. Le matrici EEG (le cui dimensioni restano invariate a $62 \times 1500$) e i relativi metadati vengono estratti dal dataset grezzo e trasferiti nel dataset pulito procedendo a blocchi iterativi (chunking).

Il risultato di questa fase è un dataset filtrato e ottimizzato, contenente esclusivamente i campioni ad alta affidabilità emotiva, pronto per le eventuali e successive operazioni di pulizia del segnale (es. rimozione degli artefatti, normalizzazione) o direttamente per il training della rete neurale.

In [5]:
# Nomi dei file
input_h5 = DB_PATH    # Dataset originale (creato nel blocco precedente)
output_h5 = '../data/processed/eeg_dataset_filtered.h5'  # Nuovo file che conterrà solo i dati pre-processati

print(f"Apertura del dataset originale: {input_h5}")

with h5py.File(input_h5, 'r') as hf_in:
    # 1. Carichiamo solo i metadati (y) in RAM, poiché sono leggeri (matrice N x 4)
    y_original = hf_in['y'][:]
    
    # Ricordiamo la struttura: [Patient_ID, Session_ID, Label, Score]
    # Lo Score è l'ultima colonna (indice 3)
    scores = y_original[:, 3]
    
    # 2. Troviamo le righe (indici) dei campioni che vogliamo mantenere (Score >= 3)
    # np.where restituisce una tupla, [0] estrae l'array di indici
    valid_indices = np.where(scores >= 3)[0]
    
    # Calcolo delle statistiche
    n_total = len(scores)
    n_valid = len(valid_indices)
    n_dropped = n_total - n_valid
    
    print(f"Campioni totali originari: {n_total}")
    print(f"Campioni da mantenere (Score >= 3): {n_valid}")
    print(f"Campioni da scartare (Score < 3): {n_dropped}")
    
    if n_valid == 0:
        print("Nessun campione supera il filtro. Operazione interrotta.")
    else:
        print(f"\nCreazione del nuovo dataset filtrato: {output_h5}")
        
        with h5py.File(output_h5, 'w') as hf_out:
            # 3. Creiamo i nuovi dataset contenitori con le dimensioni esatte (n_valid)
            X_out = hf_out.create_dataset('X', shape=(n_valid, 62, 1500), dtype='float32', chunks=True)
            y_out = hf_out.create_dataset('y', shape=(n_valid, 4), dtype='int8')
            
            # Salviamo subito i metadati aggiornati (già presenti in RAM)
            y_out[:] = y_original[valid_indices]
            
            # 4. Trasferiamo i dati EEG (X)
            # Li copiamo a "blocchi" (chunk) per non sovraccaricare la memoria del PC.
            chunk_size = 500  # Copia 500 campioni temporali alla volta
            
            print("Trasferimento dei tensori EEG in corso...")
            for i in range(0, n_valid, chunk_size):
                # Calcolo limite finale del blocco corrente
                end_idx = min(i + chunk_size, n_valid)
                
                # Estraiamo la lista degli indici originali che corrispondono a questo blocco
                batch_indices = valid_indices[i:end_idx]
                
                # h5py permette di usare una lista di indici ordinati per estrarre dati dal disco
                X_batch = hf_in['X'][batch_indices.tolist()]
                
                # Scriviamo il blocco sul nuovo disco
                X_out[i:end_idx] = X_batch
                
                print(f"  -> Progresso: copiati {end_idx}/{n_valid} campioni...")

print("\nProcesso completato! Il tuo dataset pulito è pronto in:", output_h5)

Apertura del dataset originale: ../data/processed/eeg_dataset.h5
Campioni totali originari: 14691
Campioni da mantenere (Score >= 3): 13392
Campioni da scartare (Score < 3): 1299

Creazione del nuovo dataset filtrato: ../data/processed/eeg_dataset_filtered.h5
Trasferimento dei tensori EEG in corso...
  -> Progresso: copiati 500/13392 campioni...
  -> Progresso: copiati 1000/13392 campioni...
  -> Progresso: copiati 1500/13392 campioni...
  -> Progresso: copiati 2000/13392 campioni...
  -> Progresso: copiati 2500/13392 campioni...
  -> Progresso: copiati 3000/13392 campioni...
  -> Progresso: copiati 3500/13392 campioni...
  -> Progresso: copiati 4000/13392 campioni...
  -> Progresso: copiati 4500/13392 campioni...
  -> Progresso: copiati 5000/13392 campioni...
  -> Progresso: copiati 5500/13392 campioni...
  -> Progresso: copiati 6000/13392 campioni...
  -> Progresso: copiati 6500/13392 campioni...
  -> Progresso: copiati 7000/13392 campioni...
  -> Progresso: copiati 7500/13392 campio

## Check Coerenza

In [6]:

with h5py.File(DB_PATH, 'r') as hf:
    # 1. Verifica le "chiavi" (devono essere 'X' e 'y')
    print("Dataset contenuti:", list(hf.keys()))
    
    # 2. Verifica le dimensioni (Shape)
    print("Shape di X (Segnali):", hf['X'].shape) # Deve essere (14691, 62, 1500)
    print("Shape di y (Metadati):", hf['y'].shape) # Deve essere (14691, 3)
    
    # 3. Verifica i tipi di dato
    print("Dtype di X:", hf['X'].dtype) # Deve essere float32

Dataset contenuti: ['X', 'y']
Shape di X (Segnali): (14691, 62, 1500)
Shape di y (Metadati): (14691, 4)
Dtype di X: float32


In [7]:
with h5py.File(DB_PATH, 'r') as hf:
    # Controlliamo un campione casuale o l'intero dataset se la CPU regge
    # Facciamolo a blocchi per sicurezza
    has_nan = False
    for i in range(0, hf['X'].shape[0], 1000):
        block = hf['X'][i:i+1000]
        if np.isnan(block).any():
            has_nan = True
            break
    print(f"Presenza di NaN nel dataset: {has_nan}")

Presenza di NaN nel dataset: False


In [8]:
with h5py.File(DB_PATH, 'r') as hf:
    y_emozioni = hf['y'][:, 2] # Prendi solo la colonna delle etichette
    conteggio = pd.Series(y_emozioni).value_counts().sort_index()
    print("\nDistribuzione Etichette (0:Disgust, 1:Fear, 2:Sad, 3:Neutral, 4:Happy):")
    print(conteggio)


Distribuzione Etichette (0:Disgust, 1:Fear, 2:Sad, 3:Neutral, 4:Happy):
0    2469
1    3006
2    3831
3    2955
4    2430
Name: count, dtype: int64


## File CSV con Trasformata

In [9]:

# Assicurati che il nome del file corrisponda a quello generato nel Blocco 1
h5_filename = DB_PATH 
csv_output = '../data/processed/dataset_v3_features.csv'

# Definizione bande di frequenza
BANDS = {
    'Delta': (1, 4),
    'Theta': (4, 8),
    'Alpha': (8, 14),
    'Beta':  (14, 31),
    'Gamma': (31, 50)
}

v3_data = []

print("Inizio estrazione feature (PSD ed Entropia Differenziale" \
") dai segmenti...")

with h5py.File(h5_filename, 'r') as hf:
    X = hf['X']
    y_meta = hf['y'] # Ora y_meta ha 4 colonne: [Pat, Sess, Label, Score]
    total_samples = X.shape[0]
    
    batch_size = 500
    for i in range(0, total_samples, batch_size):
        end_idx = min(i + batch_size, total_samples)
        X_batch = X[i:end_idx]
        y_batch = y_meta[i:end_idx]
        
        for j in range(X_batch.shape[0]):
            segment = X_batch[j] 
            
            # Calcolo PSD con massima risoluzione (1.5s di finestra = nperseg 1500)
            freqs, psd = welch(segment, fs=1000, nperseg=1500)
            
            # Dizionario per la riga: INCLUSIONE DELLO SCORE
            row = {
                'Patient_ID': y_batch[j, 0],
                'Session_ID': y_batch[j, 1],
                'Label':      y_batch[j, 2],
                'Score':      y_batch[j, 3]  # <--- AGGIUNTO: Quarta colonna dei metadati
            }
            
            # Estrazione potenza logaritmica (DE) per ogni canale e banda
            for ch in range(62):
                for band_name, (f_min, f_max) in BANDS.items():
                    # Maschera booleana per isolare le frequenze della banda attuale
                    idx_band = np.logical_and(freqs >= f_min, freqs <= f_max)
                    
                    # Potenza media (che equivale alla varianza sigma^2 del segnale filtrato)
                    mean_pwr = np.mean(psd[ch, idx_band])
                    
                    # Sicurezza matematica: evitiamo che la potenza sia esattamente 0
                    # per non far crashare il logaritmo con un -infinito
                    mean_pwr = max(mean_pwr, 1e-12)
                    
                    # Calcolo ESATTO della Differential Entropy (DE)
                    # DE = 0.5 * ln(2 * pi * e * mean_pwr)
                    de_value = 0.5 * np.log(2 * np.pi * np.e * mean_pwr)
                    
                    row[f'Ch{ch}_{band_name}'] = de_value
            
            v3_data.append(row)
        
        if end_idx % 1000 == 0 or end_idx == total_samples:
            print(f"Processati {end_idx}/{total_samples} segmenti...")

# Salvataggio finale
df_v3 = pd.DataFrame(v3_data)
df_v3.to_csv(csv_output, index=False)

print(f"\nDataset V3 completato e salvato in: {csv_output}")
print(f"Nuova forma della tabella: {df_v3.shape} (include la colonna Score)")

Inizio estrazione feature (PSD ed Entropia Differenziale) dai segmenti...
Processati 1000/14691 segmenti...
Processati 2000/14691 segmenti...
Processati 3000/14691 segmenti...
Processati 4000/14691 segmenti...
Processati 5000/14691 segmenti...
Processati 6000/14691 segmenti...
Processati 7000/14691 segmenti...
Processati 8000/14691 segmenti...
Processati 9000/14691 segmenti...
Processati 10000/14691 segmenti...
Processati 11000/14691 segmenti...
Processati 12000/14691 segmenti...
Processati 13000/14691 segmenti...
Processati 14000/14691 segmenti...
Processati 14691/14691 segmenti...

Dataset V3 completato e salvato in: ../data/processed/dataset_v3_features.csv
Nuova forma della tabella: (14691, 314) (include la colonna Score)
